## RFE - recursive feature elimination Classification 

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
# from sklearn.feature_selection import SelectKBest
# from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

# algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [91]:
# functions 
# feature selection
def rfe_features_classify(indep_x,dep_y,n):
    # model list 
    log_model=LogisticRegression(solver='liblinear', max_iter=1000)
    # knn_model=KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski')
    # nb_model=GaussianNB()
    # svc_model=SVC()
    dt_model=DecisionTreeClassifier(criterion='gini', splitter='best',random_state=0)
    rf_model=RandomForestClassifier(n_estimators=5, criterion='gini',random_state=0)
    rfe_model_list=[log_model  ,dt_model , rf_model]
    # rfe select features 
    rfe_result_list=[]
    for model in rfe_model_list:
        # print(model)
        rfe_alg=RFE(estimator=model,n_features_to_select=n,step=1)
        rfe_feature=rfe_alg.fit(indep_x,dep_y)
        rfe_features=rfe_feature.transform(indep_x)
        rfe_result_list.append(rfe_features)
        print('\n\nmodel: ',model, 'rfe_feature array: ',rfe_features)
        print('selected features: ', indep_x.columns[rfe_feature.get_support()])
    return rfe_result_list  

# tran test split and standardization 
def split_scaler(indep_x,dep_y):
    x_train,x_test,y_train,y_test = train_test_split(indep_x,dep_y,test_size=0.20,random_state=0)
    # standardization
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return  x_train,x_test,y_train,y_test

# model prediction & evaluation 
def cm_pred_eval(classifier,x_test):
    # prediction
    y_pred=classifier.predict(x_test)
    # confusion matrix
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    # classification report 
    from sklearn.metrics import classification_report
    classi_report=classification_report(y_test,y_pred)
    # accuracy 
    from sklearn.metrics import accuracy_score
    accuracy=accuracy_score(y_test,y_pred)
    return classifier,accuracy,cm,classi_report,x_test,y_test

### --------------- classification algortihms -----------------------------
# logistic regression 
def logistic(x_train,y_train,x_test):
    # model creation
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# KNN 
def knn(x_train,y_train,x_test):
    # model creation
    from sklearn.neighbors import KNeighborsClassifier
    classifier=KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski')
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - linear 
def svm_linear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='linear',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - Non linear 
def svm_nonlinear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='rbf',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# navie bayes
def naive(x_train,y_train,x_test):
    # model creation 
    from sklearn.naive_bayes import GaussianNB
    classifier=GaussianNB()
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# decision tree
def decision_tree(x_train,y_train,x_test):
     # model creation 
    from sklearn.tree import DecisionTreeClassifier
    classifier=DecisionTreeClassifier(criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# random forest
def random_forest(x_train,y_train,x_test):
     # model creation 
    from sklearn.ensemble import RandomForestClassifier
    classifier=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test


## ------------ final table for comparision 
def rfe_classification(rfe_resut_list):
    dataframe = pd.DataFrame(rfe_resut_list, index=['log_model','dt_model','rf_model'])
    dataframe
    return dataframe

In [106]:
data_original = pd.read_csv('preprocessed_CKD.csv')
data_ckd= data_original
data_ckd

data_ckd=pd.get_dummies(data_ckd,drop_first=True,dtype=int)
data_ckd.head()

# ip op split 
indep_x=data_ckd.drop('classification_notckd',axis=1)
dep_y = data_ckd['classification_notckd']

# k best 
no_of_features = 5
rfe_list=rfe_features_classify(indep_x,dep_y,no_of_features)
rfe_list

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

rfe_resut_list=[]
for rfe_array in rfe_list:
    x_train,x_test,y_train,y_test=split_scaler(rfe_array,dep_y)   # (selected important features , target variable) 
    rfe_row = {} 
    # algorithms 
    _,accuracy,_,_,_,_=logistic(x_train,y_train,x_test)
    rfe_row['logistic']=accuracy
    _,accuracy,_,_,_,_=knn(x_train,y_train,x_test)
    rfe_row['knn']=accuracy
    _,accuracy,_,_,_,_=svm_linear(x_train,y_train,x_test)
    rfe_row['svm_linear']=accuracy
    _,accuracy,_,_,_,_=svm_nonlinear(x_train,y_train,x_test)
    rfe_row['svm_nonlinear']=accuracy
    _,accuracy,_,_,_,_=naive(x_train,y_train,x_test)
    rfe_row['naive']=accuracy
    _,accuracy,_,_,_,_=decision_tree(x_train,y_train,x_test)
    rfe_row['decision_tree']=accuracy
    _,accuracy,_,_,_,_=random_forest(x_train,y_train,x_test)
    rfe_row['random_forest']=accuracy
    rfe_resut_list.append(rfe_row)

rfe_final_result = rfe_classification(rfe_resut_list)



model:  LogisticRegression(max_iter=1000, solver='liblinear') rfe_feature array:  [[1. 0. 1. 1. 1.]
 [4. 0. 1. 0. 0.]
 [2. 1. 0. 0. 1.]
 ...
 [0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0.]]
selected features:  Index(['al', 'rbc_normal', 'rbc_unknown', 'htn_yes', 'dm_yes'], dtype='object')


model:  DecisionTreeClassifier(random_state=0) rfe_feature array:  [[ 1.02  36.    15.4    5.2    1.   ]
 [ 1.02  18.    11.3    0.     0.   ]
 [ 1.01  53.     9.6    0.     0.   ]
 ...
 [ 1.02  26.    15.8    5.4    0.   ]
 [ 1.025 50.    14.2    5.9    0.   ]
 [ 1.025 18.    15.8    6.1    0.   ]]
selected features:  Index(['sg', 'bu', 'hemo', 'rc', 'htn_yes'], dtype='object')


model:  RandomForestClassifier(n_estimators=5, random_state=0) rfe_feature array:  [[ 1.02   1.2   15.4   44.     0.   ]
 [ 1.02   0.8   11.3   38.     0.   ]
 [ 1.01   1.8    9.6   31.     1.   ]
 ...
 [ 1.02   0.6   15.8   49.     1.   ]
 [ 1.025  1.    14.2   51.     1.   ]
 [ 1.025  1.1   15.8   53.     1.   ]]

In [94]:
rfe_final_result #5

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9125,0.9375,0.9250,0.9375,0.9500,0.9375,0.9500
dt_model,0.9875,1.0000,0.9875,1.0000,0.8875,1.0000,1.0000
rf_model,0.9750,0.9625,0.9750,0.9625,0.9375,0.9875,0.9625


In [96]:
rfe_final_result  #4

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9125,0.9375,0.9125,0.9375,0.950,0.9375,0.9500
dt_model,0.9875,1.0000,0.9875,1.0000,0.875,1.0000,1.0000
rf_model,0.9750,0.9750,0.9875,0.9750,0.950,0.9625,0.9625


In [98]:
rfe_final_result  #3

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9250,0.9250,0.9375,0.9375,0.9125,0.9375,0.9375
dt_model,0.9875,0.9875,0.9875,0.9875,0.8500,1.0000,1.0000
rf_model,0.9500,0.9375,0.9375,0.9125,0.9125,0.9750,0.9750


In [102]:
rfe_final_result  #6

,logistic,knn,svm_linear,svm_nonlinear,naive,decision_tree,random_forest
log_model,0.9125,0.9875,0.9750,0.9375,0.9500,0.9875,0.95
dt_model,0.9875,1.0000,0.9875,1.0000,0.9250,1.0000,1.00
rf_model,0.9750,0.9750,0.9750,0.9750,0.9875,1.0000,1.00
